# RSNA knee: ROI classification, inference

One streaming pass per GPU (`src/infer/pipeline.py`); the studies are split between the GPUs:

1. **Read** (DataLoader workers, the only CPU work): DICOM headers of every slice (order, geometry, Laterality tag), and the pixels of the <= 64 slices kept.
2. **Preprocess** (GPU): percentile normalisation, the 384 px cache, and the 32 x 128 x 128 volume, in the training left-right convention.
3. **Side** (GPU): left / right for *every* series, since a study can hold both knees. The model is used, not the DICOM tag.
4. **ROIs**:
   - series flipped to a right knee, then the ROI U-Net gives one box per class (ACL, MCL, menisci, the three OA compartments, Baker's), each with a confidence;
   - weak boxes are replaced by registration from the same knee's other series;
   - axial boxes are registered from the sagittal / coronal ones.
5. **Crops** (GPU), each model's own crop recipe:
   - ROI crops (8 strided slices);
   - OA compartment crops that stop at the intercondylar midline;
   - one capsule crop per series (the union of the ROI boxes) for effusion and synovitis;
   - a joint crop for contusion and fracture.
6. **Models and recipe**: the weights' `recipe.json` lists, per finding, the crop models to use; several models are combined by rank average over the test studies, and an entry can subtract another finding's score (`minus`). Findings the recipe does not cover stay at 0.5. The recipe and the models are printed below.

float16 autocast throughout. Nothing is written to disk except the per-shard CSVs.

## Imports

In [1]:
import os
import sys
import json
import time

import pandas as pd
import torch

ON_KAGGLE = os.path.exists("/kaggle/input")
if ON_KAGGLE:
    # Code and wheels come from attached datasets; the competition image lacks
    # dynamic_network_architectures (the ROI U-Net) and the DICOM decoders.
    !cp -r /kaggle/input/datasets/theoviel/rsna-knee-code/src ./
    !pip install --quiet --no-index --find-links /kaggle/input/datasets/theoviel/rsna-knee-wheels dynamic-network-architectures pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg python-gdcm
    SRC = os.path.abspath("./src")
else:
    SRC = os.path.abspath("../src")

# The pipeline runs as `<this python> -m infer.pipeline` with src/ on the path.
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, SRC)
print("src:", SRC, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      "| CPUs:", os.cpu_count())

src: /data/rsna_knee/src | GPUs: ['NVIDIA H100 80GB HBM3', 'NVIDIA H100 80GB HBM3', 'NVIDIA H100 80GB HBM3', 'NVIDIA H100 80GB HBM3'] | CPUs: 208


## Parameters

In [2]:
if ON_KAGGLE:
    DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection/"
    WEIGHTS = "/kaggle/input/datasets/theoviel/rsna-knee-weights-2/"   # output/dataset_2
    OUT_DIR = "/tmp/knee/"
else:
    DATA_DIR = "../input/"
    WEIGHTS = "../output/dataset_2/"
    OUT_DIR = "../output/inference/"
# WEIGHTS: side/ and roi/ run folders, crops/<run>/ (args.json + ckpt_final.pt), recipe.json

SERIES_DIR = "test_series"                       # <SERIES_DIR>/<StudyInstanceUID>/<SeriesInstanceUID>/*.dcm
SERIES_CSV = os.path.join(DATA_DIR, "test_series.csv")   # series -> study, plane (folders walked if absent)
GPUS = list(range(torch.cuda.device_count()))    # one process per GPU
WORKERS = max(1, os.cpu_count() // len(GPUS))    # DICOM reading processes per GPU
LIMIT = None                                     # first N studies only (debugging)

recipe = json.load(open(os.path.join(WEIGHTS, "recipe.json")))
models = {m.split("@")[0] for v in recipe.values() for m in (v if isinstance(v, list) else v["models"])}   # "name@planes": plane-restricted use of model name
for w in ("side", "roi") + tuple(f"crops/{m}" for m in models):
    assert os.path.exists(os.path.join(WEIGHTS, w, "ckpt_final.pt")), f"missing weights: {w}"
os.makedirs(OUT_DIR, exist_ok=True)
DATA_DIR, WEIGHTS, OUT_DIR = map(os.path.abspath, (DATA_DIR, WEIGHTS, OUT_DIR))
print("recipe:", recipe)

recipe: {'ACL': ['85', '88'], 'MCL': ['86', '88'], 'Medial Meniscus': ['87', '88'], 'Lateral Meniscus': ['87', '88'], 'Medial OA': ['88', '89', '90'], 'Lateral OA': ['89'], 'PF OA': ['90'], "Baker's": ['88', '91']}


## Run

In [3]:
t0 = time.time()
cmd = (f"{sys.executable} -m infer.pipeline --data_dir {DATA_DIR} --series_dir {SERIES_DIR} --series_csv {SERIES_CSV}"
       f" --weights {WEIGHTS} --out_dir {OUT_DIR} --submission submission.csv"
       f" --gpus {','.join(map(str, GPUS))} --workers {WORKERS}" + (f" --limit {LIMIT}" if LIMIT else ""))
print(cmd)
!{cmd}
print(f"{time.time() - t0:.0f}s")

/data/rsna_knee/.venv/bin/python3 -m infer.pipeline --data_dir /data/rsna_knee/input --series_dir test_series --series_csv ../input/test_series.csv --weights /data/rsna_knee/output/dataset_2 --out_dir /data/rsna_knee/output/inference --submission submission.csv --gpus 0,1,2,3 --workers 52


/data/rsna_knee/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
/data/rsna_knee/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(
/data/rsna_knee/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


shard done: 1 studies in 12s -- load 8s, warmup 1s, wait 2s, prep 0s, side 0s, unet 0s, boxes 0s, consensus 0s, crops 0s, models 0s, model 85 0s, model 86 0s, model 87 0s, model 88 0s, model 89 0s, model 90 0s, model 91 0s


shard done: 1 studies in 12s -- load 8s, warmup 1s, wait 3s, prep 0s, side 0s, unet 0s, boxes 0s, consensus 0s, crops 0s, models 0s, model 85 0s, model 86 0s, model 87 0s, model 88 0s, model 89 0s, model 90 0s, model 91 0s


shard done: 1 studies in 13s -- load 8s, warmup 1s, wait 3s, prep 0s, side 0s, unet 0s, boxes 0s, consensus 0s, crops 0s, models 0s, model 85 0s, model 86 0s, model 87 0s, model 88 0s, model 89 0s, model 90 0s, model 91 0s


-> submission.csv: 3 studies, recipe covers ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', "Baker's"] (0.5 elsewhere), 16s in total


19s


## Outputs

In [4]:
shards = sorted(d for d in os.listdir(OUT_DIR) if d.startswith("shard") and os.path.exists(os.path.join(OUT_DIR, d, "series.csv")))
series = pd.concat([pd.read_csv(os.path.join(OUT_DIR, d, "series.csv")) for d in shards], ignore_index=True)
rois = pd.concat([pd.read_csv(os.path.join(OUT_DIR, d, "rois.csv")) for d in shards], ignore_index=True)
print(series.plane.value_counts().to_dict(), "| failed:", int((series.status != "ok").sum()),
      "| sides:", series.side.value_counts().to_dict())
print(len(rois), "boxes:", rois.plane.value_counts().to_dict())
sub = pd.read_csv("submission.csv")
print(sub.shape)
sub.head()

{'Sagittal': 7, 'Axial': 4, 'Coronal': 4} | failed: 0 | sides: {'R': 10, 'L': 5}
120 boxes: {'Sagittal': 56, 'Coronal': 32, 'Axial': 32}
(3, 13)


,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.333333,1.000000,1.000000,0.500000,0.777778,0.075834,0.217075,0.5,0.5,1.000000,0.5,0.5
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,1.000000,0.333333,0.666667,0.666667,0.333333,0.469565,0.716128,0.5,0.5,0.333333,0.5,0.5
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.666667,0.666667,0.333333,0.833333,0.888889,0.258703,0.500224,0.5,0.5,0.666667,0.5,0.5
